# Day11 — Qwen2.5-VL QLoRA SFT

> **목적**: Day10에서 생성한 `adcopy_sft.jsonl`을 이용해  
> Qwen2.5-VL-7B 모델에 QLoRA SFT(스타일 정렬 파인튜닝)를 수행합니다.

## 실험 변수 (2개 고정)

| 변수 | 값 A | 값 B |
|------|------|------|
| QLoRA rank | 8 | **16** (기본값) |
| 학습 데이터 수 | 200쌍 | **350쌍** (기본값) |

## 파이프라인 흐름

```
S0  환경 설정 + GPU 확인
S1  의존성 설치
S2  adcopy_sft.jsonl 로드 + train/val 분리
S3  Qwen2.5-VL 포맷 변환 (chat template 적용)
S4  모델 + QLoRA 설정
S5  SFT 학습 실행
S6  Checkpoint 저장 + 샘플 추론 검증
S7  실험 결과 요약 (rank × 데이터 수 비교표)
```

## S0 — 환경 설정 + GPU 확인

In [ ]:
import torch

# GPU 정보 출력
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"[GPU] {gpu_name}  |  VRAM: {vram_gb:.1f} GB")
else:
    print("[WARNING] GPU 없음 — 런타임 유형을 GPU로 변경하세요")
    raise SystemExit("GPU 필수")

# ── 실험 변수 설정 ──────────────────────────────────────────
# 변수 1: QLoRA rank (8 or 16)
LORA_RANK = 16          # ← 실험 A: 8  /  실험 B: 16

# 변수 2: 학습 데이터 수 (200 or 350, None=전체)
MAX_TRAIN_SAMPLES = 350  # ← 실험 A: 200  /  실험 B: 350

# ── GPU 메모리별 배치 크기 자동 설정 ──────────────────────────
if vram_gb >= 40:         # A100
    PER_DEVICE_BATCH = 4
    GRAD_ACCUM       = 4
elif vram_gb >= 16:       # V100 / A10
    PER_DEVICE_BATCH = 2
    GRAD_ACCUM       = 8
else:                     # T4 (15GB)
    PER_DEVICE_BATCH = 1
    GRAD_ACCUM       = 16

EFFECTIVE_BATCH = PER_DEVICE_BATCH * GRAD_ACCUM
print(f"[CONFIG] rank={LORA_RANK}  max_train={MAX_TRAIN_SAMPLES}  "
      f"batch={PER_DEVICE_BATCH}  grad_accum={GRAD_ACCUM}  "
      f"effective_batch={EFFECTIVE_BATCH}")

[GPU] Tesla T4  |  VRAM: 15.6 GB
[CONFIG] rank=16  max_train=350  batch=1  grad_accum=16  effective_batch=16


## S1 — 의존성 설치

> 최초 1회만 실행. 이후 런타임 재시작 없이 진행 가능.

In [ ]:
# ── 핵심 패키지 설치 ────────────────────────────────────────
!pip install -q transformers==4.49.0
!pip install -q peft==0.14.0
!pip install -q trl==0.12.2
!pip install -q bitsandbytes==0.45.3
!pip install -q accelerate==1.2.1
!pip install -q datasets==3.2.0
!pip install -q qwen-vl-utils   # Qwen2.5-VL 이미지 전처리 유틸
!pip install -q Pillow

print("[OK] 패키지 설치 완료")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 114.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 125.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 28.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 119.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 115.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import transformers
print(transformers.__version__)  # 4.49.0 이상이어야 함

5.8.1


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ── Google Drive 마운트 (이미지 + jsonl 파일 접근) ─────────────


import os, json, random
from pathlib import Path

# !! 본인 Drive 경로에 맞게 수정 !!
DRIVE_BASE   = Path("/content/drive/MyDrive/코리아아케데미_ai_포트폴리오")
SFT_JSONL    = DRIVE_BASE / "adcopy_sft.jsonl"       # Day10 산출물
IMAGES_ROOT  = DRIVE_BASE / "images"                  # images/muji/, images/uniqlo/
OUTPUT_DIR   = DRIVE_BASE / f"checkpoints/rank{LORA_RANK}_n{MAX_TRAIN_SAMPLES}"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert SFT_JSONL.exists(),   f"[ERR] SFT jsonl 없음: {SFT_JSONL}"
assert IMAGES_ROOT.exists(), f"[ERR] images 폴더 없음: {IMAGES_ROOT}"

print(f"[OK] SFT jsonl : {SFT_JSONL}")
print(f"[OK] 이미지 루트: {IMAGES_ROOT}")
print(f"[OK] 체크포인트: {OUTPUT_DIR}")

[OK] SFT jsonl : /content/drive/MyDrive/코리아아케데미_ai_포트폴리오/adcopy_sft.jsonl
[OK] 이미지 루트: /content/drive/MyDrive/코리아아케데미_ai_포트폴리오/images
[OK] 체크포인트: /content/drive/MyDrive/코리아아케데미_ai_포트폴리오/checkpoints/rank16_n350


## S2 — adcopy_sft.jsonl 로드 + train/val 분리

In [ ]:
import random
SEED = 42
random.seed(SEED)

# ── jsonl 로드 ─────────────────────────────────────────────
with open(SFT_JSONL, encoding="utf-8") as f:
    all_records = [json.loads(l) for l in f if l.strip()]

print(f"[INFO] 전체 SFT 레코드: {len(all_records)}건")

# 스타일 분포 확인
style_dist = {}
for r in all_records:
    sid = r.get("style_id", "?")
    style_dist[sid] = style_dist.get(sid, 0) + 1
print(f"[INFO] 스타일 분포: {style_dist}")

# ── 데이터 수 제한 (실험 변수 2) ──────────────────────────────
random.shuffle(all_records)
if MAX_TRAIN_SAMPLES and len(all_records) > MAX_TRAIN_SAMPLES:
    # 스타일 균형 유지하며 샘플링
    from collections import defaultdict
    style_buckets = defaultdict(list)
    for r in all_records:
        style_buckets[r.get("style_id", "?")].append(r)
    n_styles  = len(style_buckets)
    per_style = MAX_TRAIN_SAMPLES // n_styles
    balanced  = []
    for sid, bucket in style_buckets.items():
        balanced.extend(bucket[:per_style])
    all_records = balanced[:MAX_TRAIN_SAMPLES]
    print(f"[INFO] {MAX_TRAIN_SAMPLES}쌍으로 제한 (스타일 균형 샘플링)")

# ── train / val 분리 (9:1) ────────────────────────────────
random.shuffle(all_records)
split = int(len(all_records) * 0.9)
train_records = all_records[:split]
val_records   = all_records[split:]

print(f"[OK] train={len(train_records)}  val={len(val_records)}")

[INFO] 전체 SFT 레코드: 95건
[INFO] 스타일 분포: {'A': 68, 'B': 27}
[OK] train=85  val=10


## S3 — Qwen2.5-VL 채팅 포맷 변환

SFT jsonl → Qwen2.5-VL `apply_chat_template` 입력 포맷으로 변환합니다.

```
[
  {"role": "system",    "content": [{"type": "text", "text": "..."}]},
  {"role": "user",      "content": [{"type": "image", "image": "file://..."},
                                    {"type": "text",  "text": instruction}]},
  {"role": "assistant", "content": [{"type": "text",  "text": output}]}
]
```

In [ ]:
from pathlib import Path

SYSTEM_PROMPT = (
    "당신은 한국 패션 브랜드 광고 카피라이터입니다.\n"
    "제품 이미지를 보고 지시에 따라 브랜드 스타일에 맞는 한국어 광고 카피를 "
    "한 문장으로 작성하세요."
)

def record_to_conversation(rec: dict) -> dict | None:
    """
    SFT jsonl 레코드 → Qwen2.5-VL 대화 포맷.
    이미지 경로가 존재하지 않으면 None 반환 (학습 제외).
    """
    # 이미지 경로 해석: 절대경로가 아니면 IMAGES_ROOT 기준
    img_path = Path(rec["image"])
    if not img_path.is_absolute():
        img_path = IMAGES_ROOT.parent / img_path   # Drive 기준
    if not img_path.exists():
        # fallback: IMAGES_ROOT 직접 연결
        img_path = IMAGES_ROOT / Path(rec["image"]).name
    if not img_path.exists():
        return None

    return {
        "messages": [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_PROMPT}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": f"file://{img_path.resolve()}"},
                    {"type": "text",  "text":  rec["instruction"]}
                ]
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": rec["output"]}]
            }
        ]
    }


train_convs = [r for rec in train_records if (r := record_to_conversation(rec)) is not None]
val_convs   = [r for rec in val_records   if (r := record_to_conversation(rec)) is not None]

skipped = (len(train_records) + len(val_records)) - (len(train_convs) + len(val_convs))
print(f"[OK] train={len(train_convs)}  val={len(val_convs)}  (이미지 없어 제외: {skipped}건)")

# 포맷 미리보기
print("\n=== 변환 샘플 ===")
import json
print(json.dumps(train_convs[0], ensure_ascii=False, indent=2))

[OK] train=85  val=10  (이미지 없어 제외: 0건)

=== 변환 샘플 ===
{
  "messages": [
    {
      "role": "system",
      "content": [
        {
          "type": "text",
          "text": "당신은 한국 패션 브랜드 광고 카피라이터입니다.\n제품 이미지를 보고 지시에 따라 브랜드 스타일에 맞는 한국어 광고 카피를 한 문장으로 작성하세요."
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "image",
          "image": "file:///content/drive/MyDrive/코리아아케데미_ai_포트폴리오/images/muji/bottom_008.webp"
        },
        {
          "type": "text",
          "text": "이 제품 사진을 보고 UNIQLO 스타일의 한국어 광고 카피를 한 문장으로 작성하세요."
        }
      ]
    },
    {
      "role": "assistant",
      "content": [
        {
          "type": "text",
          "text": "편안한 밴딩 허리로 하루 종일 자유롭게, 우아한 플레어 실루엣이 매일을 특별하게 만듭니다."
        }
      ]
    }
  ]
}


## S4 — 모델 + QLoRA 설정

| 항목 | 설정값 | 이유 |
|------|--------|------|
| 베이스 모델 | `Qwen/Qwen2.5-VL-7B-Instruct` | 한국어+시각 이해 동시 강함 |
| 양자화 | 4bit NF4 (bitsandbytes) | T4 16GB VRAM 내 로딩 |
| LoRA 타겟 | q,k,v,o proj + mlp gate/up/down | VL 모델 핵심 레이어 |
| rank | 8 or 16 (실험 변수) | 스타일 정렬용 적정 범위 |

In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1"

In [ ]:
import bitsandbytes
print(bitsandbytes.__version__)

0.49.2


In [ ]:
import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

# ── 4bit 양자화 설정 ───────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

# ── 모델 로드 ──────────────────────────────────────────────
print(f"[INFO] 모델 로딩: {MODEL_ID} ...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config  = bnb_config,
    torch_dtype          = torch.bfloat16,
    device_map           = "auto",
    trust_remote_code    = True,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code = True,
    min_pixels = 256 * 28 * 28,    # 이미지 해상도 범위 (Qwen2.5-VL 권장)
    max_pixels = 1280 * 28 * 28,
)

print(f"[OK] 모델 로드 완료")
print(f"[INFO] VRAM 사용: {torch.cuda.memory_allocated()/1e9:.2f} GB")

[INFO] 모델 로딩: Qwen/Qwen2.5-VL-7B-Instruct ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

[OK] 모델 로드 완료
[INFO] VRAM 사용: 5.91 GB


In [ ]:
from peft import prepare_model_for_kbit_training

# ── QLoRA 준비 ─────────────────────────────────────────────
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r              = LORA_RANK,          # 실험 변수 1
    lora_alpha     = LORA_RANK * 2,      # alpha = 2 × rank 관례
    lora_dropout   = 0.05,
    bias           = "none",
    task_type      = TaskType.CAUSAL_LM,
    target_modules = [
        # Attention
        "q_proj", "k_proj", "v_proj", "o_proj",
        # MLP
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# gradient checkpointing으로 VRAM 절약
model.enable_input_require_grads()
model.gradient_checkpointing_enable()

print(f"[OK] QLoRA 설정 완료 (rank={LORA_RANK}, alpha={LORA_RANK*2})")

trainable params: 47,589,376 || all params: 8,339,756,032 || trainable%: 0.5706
[OK] QLoRA 설정 완료 (rank=16, alpha=32)


## S5 — SFT 학습 실행

In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
from qwen_vl_utils import process_vision_info

# ── 데이터셋 객체 생성 ─────────────────────────────────────
train_dataset = Dataset.from_list(train_convs)
val_dataset   = Dataset.from_list(val_convs)

print(f"[OK] train_dataset: {len(train_dataset)}  val_dataset: {len(val_dataset)}")

# ── 콜레이터: 멀티모달 토크나이즈 ──────────────────────────
def collate_fn(examples):
    texts, image_inputs = [], []
    for ex in examples:
        msgs = ex["messages"]
        text = processor.apply_chat_template(
            msgs,
            tokenize          = False,
            add_generation_prompt = False,
        )
        texts.append(text)
        img_in, _ = process_vision_info(msgs)
        image_inputs.append(img_in)

    batch = processor(
        text   = texts,
        images = image_inputs,
        return_tensors = "pt",
        padding        = True,
        truncation     = True,
        max_length     = 512,
    )

    # labels: padding 위치는 -100으로 마스킹
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100

    # image_token도 loss 계산에서 제외 (Qwen2.5-VL 전용 처리)
    image_token_id = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
    if image_token_id is not None:
        labels[labels == image_token_id] = -100

    batch["labels"] = labels
    return batch

print("[OK] 콜레이터 정의 완료")

def clean_messages(sample):
    cleaned = []
    for msg in sample["messages"]:
        new_content = []
        if isinstance(msg["content"], list):
            for item in msg["content"]:
                clean_item = {"type": item["type"]}
                if item["type"] == "image":
                    clean_item["image"] = item["image"]
                elif item["type"] == "text":
                    clean_item["text"] = item["text"]
                new_content.append(clean_item)
        else:
            new_content = msg["content"]
        cleaned.append({"role": msg["role"], "content": new_content})
    sample["messages"] = cleaned
    return sample

train_dataset = train_dataset.map(clean_messages, load_from_cache_file=False)  # 캐시 무시
val_dataset   = val_dataset.map(clean_messages,   load_from_cache_file=False)

# ── 3. 확인 ─────────────────────────────────────────────────
import json
print(json.dumps(train_dataset[0]["messages"], indent=2, ensure_ascii=False))

[OK] train_dataset: 85  val_dataset: 10
[OK] 콜레이터 정의 완료


Map:   0%|          | 0/85 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

[
  {
    "content": [
      {
        "image": null,
        "text": "당신은 한국 패션 브랜드 광고 카피라이터입니다.\n제품 이미지를 보고 지시에 따라 브랜드 스타일에 맞는 한국어 광고 카피를 한 문장으로 작성하세요.",
        "type": "text"
      }
    ],
    "role": "system"
  },
  {
    "content": [
      {
        "image": "file:///content/drive/MyDrive/코리아아케데미_ai_포트폴리오/images/muji/bottom_008.webp",
        "text": null,
        "type": "image"
      },
      {
        "image": null,
        "text": "이 제품 사진을 보고 UNIQLO 스타일의 한국어 광고 카피를 한 문장으로 작성하세요.",
        "type": "text"
      }
    ],
    "role": "user"
  },
  {
    "content": [
      {
        "image": null,
        "text": "편안한 밴딩 허리로 하루 종일 자유롭게, 우아한 플레어 실루엣이 매일을 특별하게 만듭니다.",
        "type": "text"
      }
    ],
    "role": "assistant"
  }
]


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

# ── TrainingArguments ──────────────────────────────────────
training_args = TrainingArguments(
    output_dir                  = str(OUTPUT_DIR),
    num_train_epochs            = 3,
    per_device_train_batch_size = PER_DEVICE_BATCH,
    per_device_eval_batch_size  = PER_DEVICE_BATCH,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = 2e-4,
    lr_scheduler_type           = "cosine",
    warmup_ratio                = 0.05,
    fp16                        = False,
    bf16                        = True,   # A100/T4 모두 bfloat16
    logging_steps               = 10,
    eval_strategy               = "steps",
    eval_steps                  = 50,
    save_strategy               = "steps",
    save_steps                  = 100,
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    report_to                   = "none",   # wandb 쓰려면 "wandb"로 변경
    dataloader_num_workers      = 0,        # Colab은 0 권장
    remove_unused_columns       = False,    # 멀티모달 필수
    seed                        = SEED,
)

# ── Trainer 생성 ───────────────────────────────────────────
trainer = SFTTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    data_collator   = collate_fn,
    dataset_text_field = "messages",   # ← 이 줄 추가

    # SFTTrainer에 dataset_text_field 미사용 (콜레이터가 직접 처리)
)

print("[OK] Trainer 생성 완료")
print(f"[INFO] 총 학습 스텝: {trainer.num_training_steps() if hasattr(trainer, 'num_training_steps') else '?'}")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:309: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:328: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/

Map:   0%|          | 0/85 [00:00<?, ? examples/s]

ValueError: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples) or `list[tuple[list[str], list[str]]]` (batch of pretokenized sequence pairs).

In [ ]:
from trl import SFTTrainer, SFTConfig

# ── SFTConfig (TrainingArguments 대체) ─────────────────────
training_args = SFTConfig(
    # 출력 경로
    output_dir                  = str(OUTPUT_DIR),

    # 학습 설정
    num_train_epochs            = 3,
    per_device_train_batch_size = PER_DEVICE_BATCH,
    per_device_eval_batch_size  = PER_DEVICE_BATCH,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = 2e-4,
    lr_scheduler_type           = "cosine",
    warmup_steps                = 50,           # warmup_ratio 대신 사용
    fp16                        = False,
    bf16                        = True,

    # 로깅 / 평가 / 저장
    logging_steps               = 10,
    eval_strategy               = "steps",
    eval_steps                  = 50,
    save_strategy               = "steps",
    save_steps                  = 100,
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    report_to                   = "none",

    # 기타
    dataloader_num_workers      = 0,
    remove_unused_columns       = False,        # 멀티모달 필수
    seed                        = SEED,

    # SFTConfig 전용 설정
    max_seq_length              = 1024,         # 기본값 경고 방지
    dataset_text_field          = "text",       # 전처리 후 'text' 컬럼 사용
)

# ── messages → text 전처리 ─────────────────────────────────
def preprocess(example):
    example["text"] = processor.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return example

train_dataset = train_dataset.map(preprocess)
val_dataset   = val_dataset.map(preprocess)

# 전처리 확인
print(type(train_dataset[0]["text"]))   # <class 'str'> 이어야 함
print(train_dataset[0]["text"][:200])

# ── Trainer 생성 ───────────────────────────────────────────
trainer = SFTTrainer(
    model         = model,
    args          = training_args,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
    data_collator = collate_fn,
)

print("[OK] Trainer 생성 완료")
total_steps = (len(train_dataset) // (PER_DEVICE_BATCH * GRAD_ACCUM)) * training_args.num_train_epochs
print(f"[INFO] 총 학습 스텝 (추정): {total_steps}")

Map:   0%|          | 0/85 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

<class 'str'>
<|im_start|>system
<|vision_start|><|image_pad|><|vision_end|><|im_end|>
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|><|vision_start|><|image_pad|><|vision_end|><|im_end|>
<|im_start|>a


/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:537: UserWarning: You passed `remove_unused_columns=False` on a non-packed dataset. This might create some issues with the default collator and yield to errors. If you want to inspect dataset other columns (in this case ['text', 'messages']), you can subclass `DataCollatorForLanguageModeling` in case you used the default collator and create your own data collator in order to inspect the unused dataset columns.
  warnings.warn(


Map:   0%|          | 0/85 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

[OK] Trainer 생성 완료
[INFO] 총 학습 스텝 (추정): 15


In [ ]:
# None 이미지가 있는 샘플 찾기
for i, sample in enumerate(train_dataset):
    for msg in sample["messages"]:
        if isinstance(msg["content"], list):
            for item in msg["content"]:
                if item.get("type") == "image" and item.get("image") is None:
                    print(f"[샘플 {i}] 이미지 None 발견: {msg}")

In [ ]:
def is_valid(sample):
    for msg in sample["messages"]:
        if isinstance(msg["content"], list):
            for item in msg["content"]:
                if item.get("type") == "image":
                    # 이미지 값이 None이거나 빈 문자열이면 제거
                    if not item.get("image"):
                        return False
    return True

before = len(train_dataset)
train_dataset = train_dataset.filter(is_valid)
val_dataset   = val_dataset.filter(is_valid)

print(f"train: {before} → {len(train_dataset)} (제거: {before - len(train_dataset)}개)")

Filter:   0%|          | 0/85 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10 [00:00<?, ? examples/s]

train: 85 → 85 (제거: 0개)


In [ ]:
# 전처리 재실행
train_dataset = train_dataset.map(preprocess)
val_dataset   = val_dataset.map(preprocess)

# Trainer 재생성
trainer = SFTTrainer(
    model         = model,
    args          = training_args,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
    data_collator = collate_fn,
)

print(f"[OK] 유효 샘플 수: train={len(train_dataset)}, val={len(val_dataset)}")

Map:   0%|          | 0/85 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Map:   0%|          | 0/85 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

[OK] 유효 샘플 수: train=85, val=10


In [ ]:
# 데이터셋 구조 샘플 확인
import json
print(json.dumps(train_dataset[0]["messages"], indent=2, ensure_ascii=False))

[
  {
    "content": [
      {
        "image": null,
        "text": "당신은 한국 패션 브랜드 광고 카피라이터입니다.\n제품 이미지를 보고 지시에 따라 브랜드 스타일에 맞는 한국어 광고 카피를 한 문장으로 작성하세요.",
        "type": "text"
      }
    ],
    "role": "system"
  },
  {
    "content": [
      {
        "image": "file:///content/drive/MyDrive/코리아아케데미_ai_포트폴리오/images/muji/bottom_008.webp",
        "text": null,
        "type": "image"
      },
      {
        "image": null,
        "text": "이 제품 사진을 보고 UNIQLO 스타일의 한국어 광고 카피를 한 문장으로 작성하세요.",
        "type": "text"
      }
    ],
    "role": "user"
  },
  {
    "content": [
      {
        "image": null,
        "text": "편안한 밴딩 허리로 하루 종일 자유롭게, 우아한 플레어 실루엣이 매일을 특별하게 만듭니다.",
        "type": "text"
      }
    ],
    "role": "assistant"
  }
]


In [ ]:
def clean_messages(sample):
    cleaned = []
    for msg in sample["messages"]:
        new_content = []
        if isinstance(msg["content"], list):
            for item in msg["content"]:
                # type에 맞는 키만 남기고 null 키 제거
                clean_item = {"type": item["type"]}
                if item["type"] == "image":
                    clean_item["image"] = item["image"]
                elif item["type"] == "text":
                    clean_item["text"] = item["text"]
                new_content.append(clean_item)
        else:
            new_content = msg["content"]

        cleaned.append({"role": msg["role"], "content": new_content})

    sample["messages"] = cleaned
    return sample

train_dataset = train_dataset.map(clean_messages)
val_dataset   = val_dataset.map(clean_messages)

# 정리 확인
import json
print(json.dumps(train_dataset[0]["messages"], indent=2, ensure_ascii=False))

Map:   0%|          | 0/85 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

[
  {
    "content": [
      {
        "image": null,
        "text": "당신은 한국 패션 브랜드 광고 카피라이터입니다.\n제품 이미지를 보고 지시에 따라 브랜드 스타일에 맞는 한국어 광고 카피를 한 문장으로 작성하세요.",
        "type": "text"
      }
    ],
    "role": "system"
  },
  {
    "content": [
      {
        "image": "file:///content/drive/MyDrive/코리아아케데미_ai_포트폴리오/images/muji/bottom_008.webp",
        "text": null,
        "type": "image"
      },
      {
        "image": null,
        "text": "이 제품 사진을 보고 UNIQLO 스타일의 한국어 광고 카피를 한 문장으로 작성하세요.",
        "type": "text"
      }
    ],
    "role": "user"
  },
  {
    "content": [
      {
        "image": null,
        "text": "편안한 밴딩 허리로 하루 종일 자유롭게, 우아한 플레어 실루엣이 매일을 특별하게 만듭니다.",
        "type": "text"
      }
    ],
    "role": "assistant"
  }
]


In [ ]:
# ── 학습 실행 ──────────────────────────────────────────────
import time

print("[START] SFT 학습 시작...")
print(f"  rank={LORA_RANK}  train_samples={len(train_dataset)}  epochs=3")
print(f"  effective_batch={EFFECTIVE_BATCH}")
print()

t0 = time.time()
train_result = trainer.train()
elapsed = time.time() - t0

print(f"\n[OK] 학습 완료  ({elapsed/60:.1f}분)")
print(f"  train_loss    : {train_result.training_loss:.4f}")
print(f"  total_steps   : {train_result.global_step}")

# 메트릭 저장
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)

[START] SFT 학습 시작...
  rank=16  train_samples=85  epochs=3
  effective_batch=16



AttributeError: 'NoneType' object has no attribute 'startswith'

In [ ]:
!pip install -q -U accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 12.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.12.2 requires transformers<4.47.0, but you have transformers 5.8.1 which is incompatible.


## S6 — Checkpoint 저장 + 샘플 추론 검증

In [ ]:
# ── LoRA 어댑터 저장 ───────────────────────────────────────
ADAPTER_DIR = OUTPUT_DIR / "final_adapter"
model.save_pretrained(str(ADAPTER_DIR))
processor.save_pretrained(str(ADAPTER_DIR))

print(f"[OK] 어댑터 저장: {ADAPTER_DIR}")
print(f"[INFO] 저장 파일:")
for f in sorted(ADAPTER_DIR.iterdir()):
    print(f"  {f.name}")

In [ ]:
# ── 샘플 추론 검증 ─────────────────────────────────────────
from qwen_vl_utils import process_vision_info
from PIL import Image

model.eval()

def infer_copy(image_path: str, style: str = "MUJI") -> str:
    """
    단일 이미지 + 스타일 → 광고 카피 추론.
    style: 'MUJI' or 'UNIQLO'
    """
    instruction = (
        f"이 제품 사진을 보고 {style} 스타일의 한국어 광고 카피를 한 문장으로 작성하세요."
    )
    msgs = [
        {"role": "system",
         "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user",
         "content": [
             {"type": "image", "image": f"file://{Path(image_path).resolve()}"},
             {"type": "text",  "text": instruction},
         ]},
    ]

    text  = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    imgs, _ = process_vision_info(msgs)
    inputs  = processor(text=[text], images=imgs, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens = 64,
            do_sample      = False,   # greedy decode
            temperature    = None,
            top_p          = None,
        )

    # 입력 토큰 제거 후 디코딩
    generated = output_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(generated, skip_special_tokens=True)[0].strip()


# ── val 셋에서 3건 샘플 추론 ──────────────────────────────
print("=== 샘플 추론 결과 ===")
for rec in val_records[:3]:
    img_path = Path(rec["image"])
    if not img_path.is_absolute():
        img_path = IMAGES_ROOT.parent / img_path
    if not img_path.exists():
        continue

    style = "MUJI" if rec.get("style_id") == "A" else "UNIQLO"
    pred  = infer_copy(str(img_path), style)
    gt    = rec["output"]

    print(f"\n[Style {rec.get('style_id','?')}] {rec.get('product_id','')}")
    print(f"  GT  : {gt}")
    print(f"  Pred: {pred}")

## S7 — 실험 결과 요약

아래 셀을 **두 실험(rank=8/16 × 데이터 200/350) 모두 완료 후** 채워주세요.

| 실험 | rank | 데이터 수 | train_loss | val_loss | 예상 시간 |
|------|------|-----------|-----------|---------|----------|
| A    | 8    | 200       | _(채워주세요)_ | _(채워주세요)_ | |
| B    | 16   | 350       | _(채워주세요)_ | _(채워주세요)_ | |

In [ ]:
# ── 학습 로그 요약 ─────────────────────────────────────────
log_history = trainer.state.log_history

train_logs = [l for l in log_history if "loss" in l and "eval_loss" not in l]
eval_logs  = [l for l in log_history if "eval_loss" in l]

print(f"[실험 설정]  rank={LORA_RANK}  max_train={MAX_TRAIN_SAMPLES}")
print(f"  최종 train_loss : {train_logs[-1]['loss']:.4f}  (step {train_logs[-1]['step']})")
if eval_logs:
    best_eval = min(eval_logs, key=lambda x: x["eval_loss"])
    print(f"  최저 eval_loss  : {best_eval['eval_loss']:.4f}  (step {best_eval['step']})")

print(f"  학습 시간       : {elapsed/60:.1f}분")
print(f"  어댑터 저장     : {ADAPTER_DIR}")

# ── 결과를 JSON으로 저장 (Day12 평가 시 참조용) ──────────────
result_summary = {
    "rank"           : LORA_RANK,
    "max_train"      : MAX_TRAIN_SAMPLES,
    "train_samples"  : len(train_dataset),
    "val_samples"    : len(val_dataset),
    "train_loss"     : train_logs[-1]["loss"] if train_logs else None,
    "best_eval_loss" : min(l["eval_loss"] for l in eval_logs) if eval_logs else None,
    "elapsed_min"    : round(elapsed / 60, 1),
    "adapter_dir"    : str(ADAPTER_DIR),
    "model_id"       : MODEL_ID,
}

result_path = OUTPUT_DIR / "train_result.json"
with open(result_path, "w", encoding="utf-8") as f:
    json.dump(result_summary, f, ensure_ascii=False, indent=2)

print(f"\n[OK] 결과 저장: {result_path}")
print(json.dumps(result_summary, ensure_ascii=False, indent=2))

---

## Day12 연결 가이드

| 이 노트북 산출물 | Day12 연결 |
|---|---|
| `checkpoints/rank16_n350/final_adapter/` | Day12 평가 시 어댑터 로드 |
| `checkpoints/rank8_n200/final_adapter/`  | 실험 A 비교 어댑터 |
| `train_result.json` | 학습 결과 요약 (비교표 작성용) |

### Day12에서 할 일

```
1. hold-out 100개로 GPT-4-judge 자동 채점 (100점 스케일)
   → AS-IS(base) vs TO-BE(SFT 후) 비교
   → rank 8 vs 16 비교
2. SMART 목표 달성 여부 확인: 60/100 이상?
3. HF Hub 업로드 (어댑터 + README)
```

### 이미지 부족 시 대응 전략

현재 이미지 50~149장인 경우, 300쌍 달성 전 추가 수집이 필요합니다:

```
① 유니클로 공식몰 (uniqlo.com/kr) 카테고리별 추가 저장
   목표: tops 50장, bottoms 30장, outer 30장, 기타 40장 = 150장
② Day10 파이프라인 (S2~S9) 재실행으로 adcopy_sft.jsonl 재생성
③ 이 노트북 MAX_TRAIN_SAMPLES = 350으로 재실행
```